# Từ yêu cầu mơ hồ tới analysis plan: "khách quen không quay lại như trước"

Bộ `online-retail-ii`: cửa hàng quà tặng online ở Anh, phần lớn doanh thu từ khách sỉ (cửa hàng nhỏ) mua lặp lại.
File có hai sheet: `Year 2009-2010` (01/12/2009 tới 09/12/2010) và `Year 2010-2011` (01/12/2010 tới 09/12/2011);
hai sheet **trùng nhau** từ 01/12/2010 tới 09/12/2010.

**Bối cảnh.** Tháng 12/2011, CEO nhắn cho bạn trước kỳ lập kế hoạch năm 2012:

> *"Cảm giác của tôi là khách quen không còn quay lại nhiều như trước. Đội sales cũng nói vậy. Tìm hiểu giúp tôi,
> tôi cần biết có nên dồn ngân sách 2012 vào chương trình khách thân thiết không."*

Bạn **chưa** làm phân tích đầy đủ. Việc của bài là: (1) tính một **baseline** nhanh để biết vấn đề lớn cỡ nào và
định nghĩa có bẫy gì, (2) viết `ANALYSIS_PLAN.md` một trang gửi CEO duyệt trước khi làm.

Định nghĩa dùng chung của track: dòng hợp lệ (bỏ hoá đơn `C`, `Quantity <= 0`, `Price <= 0`), doanh thu =
`Quantity * Price`; khách = dòng có `Customer ID`; **khách mới trong tháng** = tháng của hoá đơn hợp lệ đầu tiên
của khách đó **trong dữ liệu được đưa vào**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/01-analyst-mindset/exercises/advanced/01-vague-request-analysis-plan -q` (kiểm hàm baseline và cấu trúc `ANALYSIS_PLAN.md`). Chất lượng kế hoạch chấm theo rubric trong
  lời giải.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011 (xlsx: khoảng 30-60 giây)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

RETAIL = ensure("online-retail-ii")
tx = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2010-2011")
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Baseline khách mới và khách quay lại theo tháng

`customer_baseline(tx) -> pd.DataFrame`: chỉ dùng dòng hợp lệ **có** `Customer ID`. Mỗi tháng của `InvoiceDate` một
dòng, index ngày đầu tháng (`DatetimeIndex` tên `month`, tăng dần). Sáu cột theo thứ tự:

| Cột | Định nghĩa |
| --- | --- |
| `active` | số khách có ít nhất một dòng hợp lệ trong tháng |
| `new` | số khách mà tháng này là tháng của hoá đơn hợp lệ đầu tiên (trong `tx`) |
| `returning` | `active - new` |
| `returning_share` | `returning / active` |
| `revenue` | doanh thu của khách có mã trong tháng |
| `returning_revenue_share` | phần `revenue` đến từ khách quay lại |

In [ ]:
def customer_baseline(tx: pd.DataFrame) -> pd.DataFrame:
    """Index month; cột active, new, returning, returning_share, revenue, returning_revenue_share."""
    # TODO: làm sạch; giữ dòng có Customer ID; tháng đầu tiên của mỗi khách = min tháng theo khách
    raise NotImplementedError

In [ ]:
# explore: baseline trên một sheet - đọc kỹ tháng 12/2010 và tháng 1/2011
base = customer_baseline(tx)
print(base.round(3).to_string())
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(base.index, base.returning, width=20, label="quay lại", color="#4c78a8")
ax.bar(base.index, base.new, width=20, bottom=base.returning, label="mới", color="#f58518")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Khách mới và quay lại theo tháng (chỉ sheet 2010-2011)", loc="left")
plt.show()

In [ ]:
# explore (tuỳ chọn, thêm 30-60 giây): ghép sheet 2009-2010 để khách cũ không bị đếm là "mới"
# Bỏ phần trùng 01-09/12/2010 ở sheet đầu trước khi ghép.
old = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2009-2010")
both = pd.concat([old[old.InvoiceDate < "2010-12-01"], tx], ignore_index=True)
base2 = customer_baseline(both)
compare = pd.DataFrame({"new_1_sheet": base["new"], "new_2_sheets": base2["new"]}).dropna()
print(compare.head(6).astype(int).to_string())
autumn = pd.DataFrame({
    "active_2010": base2.loc["2010-09-01":"2010-11-01", "active"].values,
    "new_2010": base2.loc["2010-09-01":"2010-11-01", "new"].values,
    "active_2011": base2.loc["2011-09-01":"2011-11-01", "active"].values,
    "new_2011": base2.loc["2011-09-01":"2011-11-01", "new"].values,
}, index=["T9", "T10", "T11"])
print(autumn.to_string())

## 2. Viết `ANALYSIS_PLAN.md`

Trước khi viết, trả lời cho chính mình:

- Chỉ dùng một sheet thì tháng 12/2010 và đầu năm 2011 "khách mới" nghĩa là gì? Xu hướng `returning_share` tăng dần
  trong năm là thật hay do cách đo? Đây là **left-censoring**: dữ liệu bị cắt đầu nên mọi khách cũ đều trông như mới.
- "Khách quen không quay lại" đo bằng metric nào thì công bằng giữa hai năm? Cùng tháng năm trước, cùng cửa sổ thời
  gian, cùng định nghĩa.
- Nếu khách quen vẫn quay lại như cũ, điều gì khác có thể tạo ra "cảm giác" của CEO và đội sales?
- Quyết định là "có dồn ngân sách 2012 vào chương trình khách thân thiết không": kết quả nào sẽ dẫn tới "có", kết quả
  nào dẫn tới "không"?

Mở `starter/ANALYSIS_PLAN.md`, điền đủ các mục theo khung (300-900 từ, mục "Giả thuyết" có ít nhất 3 giả thuyết dạng
gạch đầu dòng, xoá mọi dòng `TODO`). Kế hoạch được chấm theo rubric trong lời giải.